In [ ]:
pip install hmmlearn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Dataset

This dataset contains historical market data including:

- S&P 500 closing prices
- VIX volatility index
- Technical indicators (RSI, momentum, moving averages)

The objective is to identify hidden market regimes using a Hidden Markov Model.

In [ ]:
data = pd.read_csv("/content/processed_market_data.csv")

data.head()

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data["Date"] = pd.to_datetime(data["Date"])

In [ ]:
data = data.sort_values("Date")

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(data["Date"], data["Close"])

plt.title("S&P 500 Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.grid(True)
plt.savefig("S&P_500_Price_over_the_years.png")
plt.show()

### Feature Selection

We use the following indicators to train the Hidden Markov Model:

- returns
- volatility
- RSI
- momentum
- VIX

In [ ]:
features = [
    "returns",
    "volatility",
    "RSI",
    "momentum",
    "VIX"
]

In [ ]:
split_idx = int(len(data) * 0.8)
train_data = data.iloc[:split_idx]
test_data = data.iloc[split_idx:]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_data[features])
X_test_scaled = scaler.transform(test_data[features])

In [ ]:
X_all_scaled = scaler.transform(data[features])

### Hidden Markov Model Training

AIC → Akaike Information Criterion
BIC → Bayesian Information Criterion.

Rule:
Lower AIC/BIC = Better model


In [ ]:
from hmmlearn.hmm import GaussianHMM
n_components_range = range(2, 7)
models = []
aic_scores = []
bic_scores = []

for n in n_components_range:
    model = GaussianHMM(n_components=n, covariance_type="full", n_iter=1000, random_state=42)
    # Fit only on the training set
    model.fit(X_train_scaled)

    log_likelihood = model.score(X_train_scaled)
    n_params = n**2 + 2*n*X_train_scaled.shape[1] - 1

    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(len(X_train_scaled))

    models.append(model)
    aic_scores.append(aic)
    bic_scores.append(bic)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(n_components_range, aic_scores, label="AIC")
plt.plot(n_components_range, bic_scores, label="BIC")
plt.xlabel("Number of Regimes")
plt.ylabel("Score")
plt.title("Model Selection using AIC/BIC")
plt.grid(True)
plt.legend()
plt.savefig("Model_selection_using_AICbyBIC.png")
plt.show()

In [ ]:
best_n = n_components_range[np.argmin(bic_scores)]
print(f"Best number of regimes (lowest BIC): {best_n}")

In [ ]:
final_model = GaussianHMM(n_components=best_n, covariance_type="full", n_iter=1000, random_state=42)
final_model.fit(X_train_scaled)

In [ ]:
hidden_states = final_model.predict(X_all_scaled)

data["Regime"] = hidden_states

In [ ]:
print("\nRegime Distribution (Days):")
print(data["Regime"].value_counts())

### Regime Interpretation

The Hidden Markov Model identifies five distinct market regimes.

Regime 0 represents calm market conditions with low volatility and low VIX values.

Regime 1 and Regime 2 correspond to normal and bullish market periods with moderate volatility and positive returns.

Regime 3 represents volatile correction periods where returns decrease and volatility increases.

Regime 4 represents crisis conditions characterized by negative returns, very high volatility, and extremely high VIX values.

In [ ]:
regime_stats = data.groupby("Regime").agg({
    "returns": ["mean", "std"],
    "volatility": ["mean"],
    "VIX": ["mean"]
})

print(regime_stats)

### Visualization

In [ ]:
data.groupby("Regime")["VIX"].mean().plot(kind="bar")

plt.title("Average VIX by Market Regime")
plt.xlabel("Regime")
plt.ylabel("Average VIX")
plt.grid(True)
plt.savefig("Average_vix_by_market.png")
plt.show()

Market Regime Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))

for i in range(model.n_components):

    state = data["Regime"] == i

    plt.scatter(
        data["Date"][state],
        data["Close"][state],
        label=f"Regime {i}",
        s=10
    )

plt.title("Market Regimes Detected by HMM")

plt.xlabel("Date")
plt.ylabel("S&P 500 Price")
plt.legend()
plt.grid(True)
plt.savefig("regime_overlay_plot.png")

plt.show()

Regime Distribution Plot

In [ ]:
data["Regime"].value_counts().plot(kind="bar")

plt.title("Distribution of Market Regimes")

plt.xlabel("Regime")

plt.ylabel("Number of Days")
plt.grid(True)
plt.savefig("regime_distribution_plot")

plt.show()

Transition Matrix Visualization

In [ ]:
import seaborn as sns
import numpy as np

plt.figure(figsize=(6,5))
sns.heatmap(final_model.transmat_, annot=True, cmap="Blues")
plt.title("HMM State Transition Matrix")
plt.xlabel("Next State")
plt.ylabel("Current State")
plt.savefig("transition_matrix.png")
plt.show()

Volatility vs Regime Plot

In [ ]:
plt.figure(figsize=(14,6))
plt.scatter(data["volatility"], data["returns"], c=data["Regime"], cmap="viridis")
plt.xlabel("Volatility")
plt.ylabel("Returns")
plt.title("Market Regimes in Return-Volatility Space")
plt.colorbar(label="Regime")
plt.savefig("Volatility_vs_Regime_plot.png")
plt.show()

Market Regime Timeline Plot

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))
ax.plot(data["Date"], data["Close"], color="black")
for regime in np.unique(data["Regime"]):
    mask = data["Regime"] == regime
    ax.fill_between(data["Date"], data["Close"].min(), data["Close"].max(),
                    where=mask, alpha=0.2, label=f"Regime {regime}")
ax.set_title("HMM Market Regime Detection Timeline")
ax.set_xlabel("Date")
ax.set_ylabel("S&P 500")
ax.legend()
plt.savefig("Market_regime_timeline.png")
plt.show()


Regime statistics

In [ ]:
regime_stats = data.groupby("Regime").agg({
    "returns": ["mean", "std"],
    "volatility": ["mean"],
    "Close": ["count"]
})

regime_stats.columns = [
    "Avg_Return",
    "Return_STD",
    "Avg_Volatility",
    "Days_in_Regime"
]

print(regime_stats)

In [ ]:
import matplotlib.pyplot as plt

regime_stats["Avg_Return"].plot(kind="bar", figsize=(8,5))

plt.title("Average Returns per Market Regime")
plt.xlabel("Regime")
plt.ylabel("Average Return")
plt.grid(True)
plt.savefig("Average_returns_per_market_regime.png")
plt.show()

In [ ]:
regime_stats["Avg_Volatility"].plot(kind="bar", figsize=(8,5))

plt.title("Average Volatility per Market Regime")
plt.xlabel("Regime")
plt.ylabel("Volatility")
plt.grid(True)
plt.savefig("Average_volatility_per_market_regime.png")
plt.show()

In [ ]:
import seaborn as sns

sns.boxplot(x="Regime", y="RSI", data=data)

plt.title("RSI Distribution Across Market Regimes")
plt.savefig("RSI_Distribution_across_market_regimes.png")
plt.show()

Combined Market Regime Visualization:

-Black Line → S&P 500 price
-Red Line → VIX (market fear)
-Colored points → Market regimes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, ax1 = plt.subplots(figsize=(15,6))

ax1.plot(data["Date"], data["Close"], color="black", label="S&P 500")
ax1.set_ylabel("S&P 500 Price")

scatter = ax1.scatter(
    data["Date"],
    data["Close"],
    c=data["Regime"],
    cmap="viridis",
    s=8,
    label="Market Regime"
)

ax2 = ax1.twinx()

ax2.plot(
    data["Date"],
    data["VIX"],
    color="red",
    linestyle="dashed",
    label="VIX"
)

ax2.set_ylabel("VIX")

plt.title("S&P 500, VIX, and Market Regimes")

fig.legend(loc="upper left")
plt.grid(True)
plt.xticks(rotation=45)
plt.savefig("sp500_vix_regimes.png", dpi=300)
plt.show()

In [ ]:
data["Regime"].value_counts().sort_index().plot(
    kind="bar",
    title="Number of Days in Each Market Regime"
)

plt.xlabel("Regime")
plt.ylabel("Number of Days")
plt.grid(True)
plt.savefig("Number_of_days_in_each_market_regime.png")
plt.show()

In [ ]:
import seaborn as sns

sns.boxplot(x="Regime", y="returns", data=data)

plt.title("Returns Distribution Across Market Regimes")

plt.show()

#Regime aware investing
A simple trading rule:

-If regime = Bull or Calm → Invest in S&P500

-If regime = Crisis or Volatile → Stay in cash

In [ ]:
regime_risk = data.groupby("Regime")["VIX"].mean().sort_values()

In [ ]:
safe_regimes = regime_risk.head(3).index.tolist()
print(f"\nDynamically identified Safe Regimes (Lowest VIX): {safe_regimes}")

In [ ]:
data["signal"] = 0
data.loc[data["Regime"].isin(safe_regimes), "signal"] = 1

In [ ]:
data["strategy_returns"] = data["signal"].shift(1) * data["returns"]

In [ ]:
data["market_cum"] = (1 + data["returns"]).cumprod()
data["strategy_cum"] = (1 + data["strategy_returns"]).cumprod()

In [ ]:
# Plot Strategy vs Buy & Hold
plt.figure(figsize=(14,6))
plt.plot(data["Date"], data["market_cum"], label="Buy & Hold (Market)")
plt.plot(data["Date"], data["strategy_cum"], label="HMM Regime Strategy")
plt.title("HMM Regime-Based Trading Strategy vs. Buy & Hold")
plt.xlabel("Date")
plt.ylabel("Cumulative Returns")
plt.legend()
plt.grid(True)
plt.savefig("HMM_Regime_Based_Trading_Strategy.png")
plt.show()

Evaluation

In [ ]:
# Calculate trading days per year (approx 252)
days = len(data)
years = days / 252

In [ ]:
# Annualized Returns
market_ann = (data["market_cum"].iloc[-1]) ** (1 / years) - 1
strategy_ann = (data["strategy_cum"].iloc[-1]) ** (1 / years) - 1

In [ ]:
print("Market Return:",
      data["market_cum"].iloc[-1])

print("Strategy Return:",
      data["strategy_cum"].iloc[-1])

In [ ]:
# Annualized Sharpe Ratio (Assuming 0% risk-free rate for simplicity)
market_sharpe = (data["returns"].mean() / data["returns"].std()) * np.sqrt(252)
strategy_sharpe = (data["strategy_returns"].mean() / data["strategy_returns"].std()) * np.sqrt(252)

In [ ]:
print("Market Sharpe Ratio:",market_sharpe)
print("Strategy Sharpe Ratio:", strategy_sharpe)

In [ ]:
def max_drawdown(series):
    cum = (1 + series).cumprod()
    peak = cum.cummax()
    drawdown = (cum - peak) / peak
    return drawdown.min()

print("\n--- Strategy Performance Evaluation ---")
print(f"Total Market Return:      {data['market_cum'].iloc[-1] - 1:.2%}")
print(f"Total Strategy Return:    {data['strategy_cum'].iloc[-1] - 1:.2%}")
print(f"Annualized Market Return: {market_ann:.2%}")
print(f"Annualized Strat Return:  {strategy_ann:.2%}")
print(f"Market Sharpe Ratio:      {market_sharpe:.2f}")
print(f"Strategy Sharpe Ratio:    {strategy_sharpe:.2f}")
print(f"Market Max Drawdown:      {max_drawdown(data['returns']):.2%}")
print(f"Strategy Max Drawdown:    {max_drawdown(data['strategy_returns']):.2%}")

In [ ]:
cum_market = (1 + data["returns"]).cumprod()
cum_strategy = (1 + data["strategy_returns"]).cumprod()

market_drawdown = cum_market / cum_market.cummax() - 1
strategy_drawdown = cum_strategy / cum_strategy.cummax() - 1

plt.figure(figsize=(14,6))
plt.plot(data["Date"], market_drawdown, label="Market Drawdown", alpha=0.7)
plt.plot(data["Date"], strategy_drawdown, label="Strategy Drawdown", alpha=0.9)
plt.title("Drawdown Comparison: Strategy vs. Market")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.savefig("Drawdown_comparison.png")
plt.show()

In [ ]:
import joblib

# Save the trained HMM and the Scaler
joblib.dump(final_model, 'hmm_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("Model and scaler saved successfully for deployment.")

The regime-based strategy avoids investing during high-volatility
and crisis regimes detected by the Hidden Markov Model.

This reduces exposure to large market drawdowns and improves
risk-adjusted performance compared to a simple buy-and-hold strategy.

## Conclusion

In this project, a Hidden Markov Model (HMM) was applied to financial market data to identify hidden market regimes in the S&P 500 index. The dataset combined historical S&P 500 price data with the CBOE Volatility Index (VIX), which acts as a measure of market fear and uncertainty. Several technical indicators such as returns, volatility, momentum, moving averages, and RSI were engineered to capture different aspects of market behavior.

Using these features, the HMM successfully identified multiple distinct market regimes characterized by different levels of returns and volatility. The analysis showed that certain regimes correspond to stable market conditions with low volatility, while others represent highly volatile or crisis periods. The transition matrix further illustrated how markets move between these regimes over time.

To evaluate the practical usefulness of regime detection, a simple regime-based trading strategy was implemented. The strategy reduced exposure during high-volatility regimes and invested during stable regimes. Although the strategy produced slightly lower total returns compared to a buy-and-hold approach, it significantly reduced maximum drawdown, demonstrating improved risk management during turbulent market periods.

Overall, the results show that Hidden Markov Models are effective tools for identifying hidden market structures and regime shifts in financial time series. Such regime detection models can help investors and analysts better understand market dynamics and design more robust investment strategies. In future work, the detected market regimes can be integrated with deep learning models such as LSTM to build regime-aware prediction systems for financial forecasting.
